In [1]:
# Cell 1 — Setup
import polars as pl
from Python.pipeline import games
from Python import config

In [2]:
# Cell 2 — Run Level 1 build
paths = games.run(refresh_player_map=False)
paths

[level 1] wrote player_id_map: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\dimensions\player_id_map.parquet
[level 1] wrote pitcher_games: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\pitcher_games.parquet
[level 1] wrote batter_games: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\batter_games.parquet
[level 1] wrote park_factors: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\park_factors.parquet


{'player_id_map': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/dimensions/player_id_map.parquet'),
 'pitcher_games': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/processed/pitcher_games.parquet'),
 'batter_games': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/processed/batter_games.parquet'),
 'park_factors': WindowsPath('C:/Users/ckaplinger/Downloads/Personal-Projects/MLB-Props/Data/processed/park_factors.parquet')}

In [3]:
# Cell 3 — Load and inspect each output
pitcher_games = pl.read_parquet(config.PITCHER_GAMES_PATH)
batter_games = pl.read_parquet(config.BATTER_GAMES_PATH)
park_factors = pl.read_parquet(config.PARK_FACTORS_PATH)

pitcher_games.shape, batter_games.shape, park_factors.shape

((14124, 137), (146385, 46), (120, 5))

In [4]:
# Cell 4 — Confirm k_rate label sanity (no divide-by-zero artifacts)
pitcher_games.select("k_rate").describe()

statistic,k_rate
str,f64
"""count""",14124.0
"""null_count""",0.0
"""mean""",0.219371
"""std""",0.105873
"""min""",0.0
"""25%""",0.142857
"""50%""",0.210526
"""75%""",0.285714
"""max""",0.722222


In [5]:
# Cell 5 — Confirm park_factors includes the projected "next season"
park_factors.select(pl.col("season").unique().sort())

season
i32
2023
2024
2025
2026


In [6]:
# Cell 6 — Verify the player_id_map write claim
import os
map_path = config.PLAYER_ID_MAP_PATH
os.path.getmtime(map_path)  # compare against run() execution time to confirm a real write happened

1784830202.8895142

In [7]:
# Cell 7 — Batter name enrichment coverage check
batter_games.select(pl.col("batter_name").is_null().sum().alias("unmapped_batters"))

unmapped_batters
u32
0
